In [ ]:
# Exploration: TBI cohort and ICU admission

# 1. Load and inspect the seven tables
# 2. Build TBI cohort
# 3. Define outcome (ICU within 24 hours of arrival at ED)
# 4. Explore triage variables and missingness
# 5. Compare ICU vs non-ICU patients
# 6. Save the cohort

In [ ]:
# Set up

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

ED = "../data/mimic-iv-ed-2.2/ed/"
HOSP = "../data/mimic-iv-2.2/hosp/"
ICU = "../data/mimic-iv-2.2/icu/"

admitted_only = True  # False = all ED TBI visits & dischared patients are "no ICU"
icu_windows_hours = 24 # outcome window
vitals = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]

In [ ]:
# Load and inspect tables

edstays    = pd.read_csv(ED + "edstays.csv.gz", parse_dates=["intime", "outtime"])
triage     = pd.read_csv(ED + "triage.csv.gz", dtype={"pain": str})
vitalsign  = pd.read_csv(ED + "vitalsign.csv.gz", parse_dates=["charttime"], dtype={"pain": str})
diagnosis  = pd.read_csv(ED + "diagnosis.csv.gz", dtype={"icd_code": str})
patients   = pd.read_csv(HOSP + "patients.csv.gz")
admissions = pd.read_csv(HOSP + "admissions.csv.gz", parse_dates=["admittime", "dischtime"])
icustays   = pd.read_csv(ICU + "icustays.csv.gz", parse_dates=["intime", "outtime"])

tables = {"edstays": edstays, "triage": triage, "vitalsign": vitalsign, "diagnosis": diagnosis,
          "patients": patients, "admissions": admissions, "icustays": icustays}

In [ ]:
def inspect(df, name):
    """Print quick overview of table with size, column types, missing values."""

    # Rows = records, unique subject_id = distinct patients
    # If rows > patients, some patients appear more than once
    
    print(f"{name}: {len(df):,} rows, {df['subject_id'].nunique():,} patients")
    
    
    # Tranposed to read across page
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "% missing": (df.isna().mean() * 100).round(1),
        "unique values": df.nunique(),
    }).T)
    display(df.head())

for name, df in tables.items():
    inspect(df, name)

In [ ]:
# Comments

# edstays: one row per ED visit and hadm_id is empty when patient is not admitted
# triage: one row per ED visit, no time stamp
# vitalsign: repeated readings during ED stay, readings after triage were not available at triage decision
# diagnosis: codes assigned after visit, used to define cohort, not for prediction
# patients: dates shifted for patient privacy, age calculated by anchor_age + (visit year - anchor_year)
# admissions, icustays: describe after ED, only supply outcome

In [ ]:
# Data integrity checks
# Fails indicate merging will duplicate or lose patient records

# Each ED visit (stay_id) should appear once in edstays
print("edstays one row per stay:", edstays["stay_id"].is_unique)

# Each ED visit (stay_id) should appear once in triage so merging does not duplicate visits
print("triage one row per stay:", triage["stay_id"].is_unique)

# Every admitted ED visit (non-empty hadm_id) should link to hospital admission
print("ED hadm_ids found in admissions:", f"{edstays['hadm_id'].dropna().isin(admissions['hadm_id']).mean():.1%}")

# GCS is not recorded in MIMIC-IV-ED, confirmed below so can be stated as limitation
print("GCS column recorded in triage:", [c for t in (triage, vitalsign) for c in t.columns if "gcs" in c.lower()] or "none")

In [ ]:
# TBI cohort building

# ICD versions
# The database runs from 2011-2019. US hospitals switched from ICD-9 to ICD-10
# coding in October 2015, so the cohort requires both coding systems to cover
# the entire period.

icd = diagnosis["icd_code"].str.strip()

# ICD 9 - 850-854 are intracranial injuries (concussion, contusion, haemorrhage)
tbi9 = (diagnosis["icd_version"] == 9) & icd.str[:3].isin(["850", "851", "852", "853", "854"])

# ICD 10 - S06 is intracranial injury, the equivalent of ICD-9 850-854
# and A limits the cohort to initial encounters (not follow-up visits)
tbi10 = (diagnosis["icd_version"] == 10) & icd.str.startswith("S06") & icd.str.endswith("A")

print("TBI diagnosis rows: ICD-9", tbi9.sum(), "and ICD-10", tbi10.sum())
print("ED stays with ICD-9 TBI code: ", diagnosis.loc[tbi9]["stay_id"].nunique())
print("ED stays with ICD-10 TBI code: ", diagnosis.loc[tbi10]["stay_id"].nunique())

In [ ]:
# Attritive count of ED stays at each stage
# Will form the patient-flow attrition diagram in the report

attrition = [] # one entry per step
def step(df, label):
    attrition.append({"step": label, "ED stays": len(df), "patients": df["subject_id"].nunique()})
    print(f"{label:<45} {len(df):>7,} stays")

step(edstays, "All ED stays")

# isin() ensures each visit is counted once even with multiple TBI codes present
cohort = edstays[edstays["stay_id"].isin(diagnosis.loc[tbi9 | tbi10]["stay_id"])].copy()
step(cohort, "With a TBI diagnosis code")

# Age shifted to protect privacy, calculated by anchor_age + (visit year - anchor_year) and filtered to adults only
# Ages above 89 are recorded as 91 in MIMIC, so oldest ages are approximate
cohort = cohort.merge(patients[["subject_id", "anchor_age", "anchor_year"]], on="subject_id", how="left")
cohort["age"] = cohort["anchor_age"] + (cohort["intime"].dt.year - cohort["anchor_year"])
cohort = cohort[cohort["age"] >= 18].copy() # missing ages fail >= 18 filter
step(cohort, "Adults (18+) with known age")

# admitted_only = False established in set up to include discharged patients as "no ICU"
if admitted_only:
    cohort = cohort[cohort["hadm_id"].notna()].copy()
    step(cohort, "Admitted to hospital")

# Ensure merges have not caused duplication
assert cohort["stay_id"].is_unique

In [ ]:
# ICU admissions outcome
# After triage within 24 hours of ED arrival
# Defines label but not used for model features -> leakage

# Uses first ICU stay of the linked hospital admission as one admission may contain multiple ICU stays
first_icu = (icustays.groupby("hadm_id", as_index = False)["intime"].min()
             .rename(columns = {"intime": "icu_intime"}))
cohort = cohort.merge(first_icu, on = "hadm_id", how = "left")

# Hours from ED arrival to first ICU admission, empty for no ICU patients
cohort["hours_to_icu"] = (cohort["icu_intime"] - cohort["intime"]).dt.total_seconds() / 3600

# icu_any: ICU at any point in admission
cohort["icu_any"] = cohort["icu_intime"].notna().astype(int)
# icu_24h: ICU within 24 hour window
# a later ICU admission may reflect deterioration on ward, which triage data cannot predict
cohort["icu_24h"] = cohort["hours_to_icu"].between(0, icu_windows_hours).astype(int)

print(f"ICU at any point: {cohort['icu_any'].sum():>5} ({cohort['icu_any'].mean():.1%})")
print(f"ICU within {icu_windows_hours} hours: {cohort['icu_24h'].sum():>5} ({cohort['icu_24h'].mean():.1%})")
print(f"ICU after {icu_windows_hours} hours: {(cohort['icu_any'] - cohort['icu_24h']).sum():>5}")
display(cohort["hours_to_icu"].describe(percentiles=[0.25, 0.5, 0.75, 0.90]).round(1).to_frame("hours from ED arrival to ICU"))


In [ ]:
# ICU timings: report figure

h = cohort["hours_to_icu"].dropna() # ICU patients only
pct_in_window = (h <= icu_windows_hours).mean() * 100 # percentage of ICU admissions within window

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# A: distribution over first 48 hours in one hour bins, with later admissions noted in text
ax1.hist(h[h <= 48], bins = 48, range = (0, 48), color = "#3b6ea5")
ax1.axvline(icu_windows_hours, ls = "--", c = "k", lw = 1)
ax1.text(0.97, 0.95, f"{(h > 48).sum()} admissions after \n48 hours not shown",
         transform = ax1.transAxes, ha = "right", va = "top")
ax1.set(xlabel = "Hours from ED arrival to ICU", ylabel = "ICU admissions",
        title = "A: Distribution of ICU admission times", xlim = (0, 48))

# B: cumulative percentage - ICU admissions by the hour
# Values are not capped - the x-axis is limited so late admissions don't create a false spike at the end
hs = np.sort(h)
ax2.plot(hs, np.arange(1, len(hs) + 1) / len(hs) * 100, c = "#3b6ea5", lw = 2)
ax2.axvline(icu_windows_hours, ls = "--", c = "k", lw = 1)
ax2.axhline(pct_in_window, ls = ":", c = "grey", lw = 1)
ax2.text(icu_windows_hours + 1.5, pct_in_window - 9,
         f"{pct_in_window:.1f}% within {icu_windows_hours} h", fontsize = 9)
ax2.set(xlabel = "Hours from ED arrival to ICU", ylabel = "Cumulative % of ICU admissions",
        title = "B: Cumulative ICU admissions", xlim = (0, 72), ylim = (0, 102))

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig("../results/fig_icu_timing.png", dpi=300, bbox_inches="tight")   # high-res copy for the report
plt.show()

In [ ]:
# Triage variables and missingness

# Triage has one row per ED stay, so merge adds triage columns without duplication of any visits
cohort = cohort.merge(triage.drop(columns = "subject_id"), on = "stay_id", how = "left")

# Flag remains for rows where entire triage form is empty (all six vitals missing)
# Patients likely bypassed normal triage
cohort["all_vitals_missing"] = cohort[vitals].isna().all(axis=1)
assert cohort["stay_id"].is_unique

# Percentage missing per variable - overall and by outcome
# If data is missing at random, ICU and non-ICU columns would be similar
# Large differences indicates missingness carries information about the outcome
miss = cohort[vitals + ["pain", "acuity", "chiefcomplaint"]].isna()
display(pd.DataFrame({
    "% missing": miss.mean() * 100,
    "% missing, no ICU": miss[cohort["icu_24h"] == 0].mean() * 100,
    "% missing, ICU": miss[cohort["icu_24h"] == 1].mean() * 100,
}).round(1))

In [ ]:
# Locating centirely blank triage rows
# Tests whether blank triage marks most critically ill patients

# Number of stays and ICU rate in each group
g = cohort.groupby("all_vitals_missing")
display(g["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# Acuity (ESI: 1 = most urgent, 5 = least)
# Normalise = "index" gives row percentages so each row will sum to 1
# Missing acuity kept as own column
print("Acuity (row percentage):")
display(pd.crosstab(cohort["all_vitals_missing"], cohort["acuity"].fillna("missing"), normalize = "index").round(2))

# Arrival mode shows what percentage of arrival type had blank triage
# Arrival mode is known at triage point
print("Arrival mode (row percentage):")
display(pd.crosstab(cohort["arrival_transport"], cohort["all_vitals_missing"], normaliz = "index").round(2))

In [ ]:
# Explore later vitals testing
# Vitals may be charted in vitalsign table instead of triage form
# Readings soon after arrival may be considered "at triage" and may fill gaps

# Minutes from ED arrival to each vital sign reading
vs = vitalsign.merge(edstays[["stay_id", "intime"]], on="stay_id")
vs["mins_after_arrival"] = (vs["charttime"] - vs["intime"]).dt.total_seconds() / 60

# Readings timestamped before ED arrival are impossible (assumed errors)
print("Readings before arrival (excluded):", (vs["mins_after_arrival"] < 0).sum())
vs_ok = vs[vs["mins_after_arrival"] >= 0]

# Time of first valid reading for each stay
cohort["first_vitals_mins"] = cohort["stay_id"].map(vs.groupby("stay_id")["mins_after_arrival"].min())

# Stays with no valid vitalsign readings
# (lambda s: s.isna().sum() counts missing values within groups)
print("\nNo vitalsign readings at all:")
display(cohort.groupby("all_vitals_missing")["first_vitals_mins"].apply(lambda s: s.isna().sum()))

# Explore when blank triage patients receive readings within each time window
blank = cohort["all_vitals_missing"]
for cut in (15, 30, 60):
    covered = blank & cohort["first_vitals_mins"].between(0, cut)
    print(f"Within {cut:>2} minutes of ED arrival: {covered.sum()} of {blank.sum()} blank-triage stays")

In [ ]:
# Exploration of data quality - implausible readings and free-text pain scores
# 1st and 99th percentiles show realistic range
# Min and max show recording errors that require removal before modelling

display(cohort[vitals + ["acuity"]].describe(percentiles=[.01, .5, .99]).T.round(1))

# Pain is free text split into plain numbers, missing, or other texts
pain_num = pd.to_numeric(cohort["pain"], errors = "coerce") # non-numerical text -> NaN
print("pain: numeric", f"{pain_num.notna().mean():.1%}",
      "| missing", f"{cohort['pain'].isna().mean():.1%}",
      "| other text", f"{(pain_num.isna() & cohort['pain'].notna()).mean():.1%}")

# Most common non-numeric entries
# .Str.Strip() removes stray spaces, otherwise "unable" and "unable " would be counted as different values
display(cohort.loc[pain_num.isna() & cohort["pain"].notna(), "pain"].str.lower().value_counts().head(15))


In [ ]:
# Clean physiologically implausible values
# Ranges are generous to remove clear recording errors and keep abnormal but real values
# Temperature is mostly Fahrenheit but some recorded in Celsius

# Values between 30 and 45 are only plausible as Celsius, so are concerted to Fahrenheit
celsius = cohort["temperature"].between(30, 45)
print("Temperature converted from Celsius:", celsius.sum())
cohort.loc[celsius, "temperature"] = cohort.loc[celsius, "temperature"] * 9 / 5 + 32

plausible = {
    "temperature": (90, 110),
    "heartrate":   (20, 250),
    "resprate":    (4, 60),
    "o2sat":       (50, 100),
    "sbp":         (40, 300),
    "dbp":         (20, 200),
}

for col, (low, high) in plausible.items():
    bad = cohort[col].notna() & ~cohort[col].between(low, high)
    print(f"{col:<12} {bad.sum():>3} implausible values removed: {sorted(cohort.loc[bad, col].tolist())}")
    cohort.loc[bad, col] = np.nan

In [ ]:
# Assess pain score by converting into two features
# Many entries are non-numerical and mean "unable to assess"

# uta / ua / u/a / u / unable = unable to assess
# ett / intubated = breathing tube in place
# c / crit / critical = assumption of critical patient

# Patient who cannot report pain usually has reduced consciousness

pain_text = cohort["pain"].astype(str).str.strip().str.lower()
cohort["pain_unable"] = pain_text.str.match(r"^(uta|ua|u/a|u|unable|ett|intubated|crit|critical|c)$").astype(int)

# Numeric pain score 0-10
# Text becomes NaN, values outside 0-10 become errors
cohort["pain_score"] = pd.to_numeric(cohort["pain"], errors="coerce")
cohort.loc[(cohort["pain_score"] < 0) | (cohort["pain_score"] > 10), "pain_score"] = np.nan

print(cohort.groupby("pain_unable")["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# Flag blank triage patients who bypass standard triage
cohort["triage_bypassed"] = cohort["all_vitals_missing"].astype(int)

# Explore relation of pain_unable to triage_bypassed
print(cohort.groupby(["triage_bypassed", "pain_unable"])["icu_24h"].agg(stays = "count", icu_rate = "mean").round(3))

In [ ]:
# Determine ICU vs non-ICU patients
# SMD measures how far apart two groups are by SD, indepedent of sample size
# |SMD| > 0.1 = meaningful difference, > 0.5 large
# Vitals can only be compared where recorded, i.e., among most normally triaged patients

# Explore demographics
numeric = ["age"] + vitals + ["pain_score", "acuity"]
icu, no_icu = cohort[cohort["icu_24h"] == 1], cohort[cohort["icu_24h"] == 0]

def smd(a, b):
    """ Standardised mean difference between two groups (ignore missing values)"""
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

display(pd.DataFrame({
    "ICU median": icu[numeric].median(),
    "no ICU median": no_icu[numeric].median(),
    "SMD": [smd(icu[c], no_icu[c]) for c in numeric],
}).round(2))

# Categorical variables: ICU rate within categories
for col in ["gender", "arrival_transport"]:
    display(cohort.groupby(col)["icu_24h"].agg(stays="count", icu_rate="mean").round(3))

# Binary flags established - percentage of each outcome group with flag
flags = ["triage_bypassed", "pain_unable"]
display(pd.DataFrame({
    "Percentage of ICU": icu[flags].mean() * 100,
    "Percentage of no ICU": no_icu[flags].mean() * 100
}).round(1))

In [ ]:
# Triage variables by outcome: report boxplots figure
# Boxes show median and interquartile range, whiskers typical range
# Outliers hidden by showfliers = False so boxes remain readable
# Patients with recorded values are included

labels = {   # titles with units
    "age": "Age (years)", "temperature": "Temperature (°F)", "heartrate": "Heart rate (bpm)",
    "resprate": "Respiratory rate (/min)", "o2sat": "O₂ saturation (%)",
    "sbp": "Systolic BP (mmHg)", "dbp": "Diastolic BP (mmHg)",
    "pain_score": "Pain score (0–10)", "acuity": "ESI acuity (1 = most urgent)",
}

fig, axes = plt.subplots(3, 3, figsize = (11, 9))
for ax, col in zip(axes.flatten(), numeric):
    bp = ax.boxplot([icu[col].dropna(), no_icu[col].dropna()],
                    showfliers = False, patch_artist = True, widths = 0.6, medianprops = dict(color = "black"))
    for box, colour in zip(bp["boxes"], ["#c0392b", "#7f8c99"]):   # ICU red, no ICU grey
        box.set_facecolor(colour)
        box.set_alpha(0.6)
    ax.set_xticks([1, 2], ["ICU", "No ICU"])
    ax.set_title(labels[col], fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
fig.savefig("../results/fig_triage_boxplots.png", dpi = 300, bbox_inches = "tight")
plt.show()



In [ ]:
# Explore base-line of how well nurse-assigned acuity alone predicts ICU admission
# ESI = Emergency Severity Index, used to assess acuity where 1 = most urgent and 5 = least urgent
# Models must clearly outperform this to add value beyond triage nurses' assessments

from sklearn.metrics import roc_auc_score

has_esi = cohort["acuity"].notna() # AUROC cannot use missing values

# Minus sign: roc_auroc_socre expects higher scores for positive class but lower ESI means more urgent
# Therefore the scale is flipped
print("AUROC of ESI acuity alone:", round(roc_auc_score(cohort.loc[has_esi, "icu_24h"], -cohort.loc[has_esi, "acuity"]), 3))

# ICU rate at each ESI level
display(cohort.groupby("acuity")["icu_24h"].agg(stays = "count", icu_rate = "mean").round(3))

# Second baseline
# Fairer baseline for full cohort - missing acuity = bypassed triage = most urgent -> ESI = 0
esi_filled = cohort["acuity"].fillna(0)
print("AUROC of ESI (missing = most urgent), full cohort:",
      round(roc_auc_score(cohort["icu_24h"], -esi_filled), 3))                                    

In [ ]:
# Summarise and save exploration

# Patient-flow table recorded at each cohort building step
# Becomes flow diagram in report
attrition_table = pd.DataFrame(attrition)
display(attrition_table)
attrition_table.to_csv("../results/attrition.csv", index = False)

# Save cleaned cohort for preprocessing and modelling scripts
# Saved inside data/, which .gitignore excludes as it contains patient-level data.
import os
os.makedirs("../data/interim", exist_ok = True)
cohort.to_pickle("../data/interim/cohort.pkl")
print("Saved", len(cohort), "stays to data/interim/cohort.pkl")

# Final check for repeat visits
# Some patients have several TBI ED stays
# Train/test sets must be split by patient (subject_id) so no patient appears in both
visits = cohort.groupby("subject_id").size()
print("Patients:", len(visits), "| with more than one TBI visit:", (visits > 1).sum())

In [ ]:
# Feature development

# Chief complaint terms

# Terms explored for patients in training set and features are chosen by their ICU rate is a modelling decision
# Therefore testing set is not involved

train = pd.read_pickle("../data/processed/train.pkl")
train_cohort = cohort[cohort["subject_id"].isin(train["groups"])]
print("Training stays used:", len(train_cohort)) # should print 1824

# Split complaints into their separate terms eg Fall, head injury becomes "fall", "head injury"
# Protects "/" so that the common term "s/p" (status post) is not split on the slash
text = train_cohort["chiefcomplaint"].str.lower().str.replace("s/p", "sp", regex=False)
words = text.str.split(r"[,/;]+").explode().str.strip()
cc = pd.DataFrame({"term": words, "icu": train_cohort.loc[words.index, "icu_24h"]})

# Terms that appear in > 20 stays in order of commonality and with ICU rate
display(cc.groupby("term")["icu"].agg(stays="count", icu_rate="mean")
        .query("stays >= 15").sort_values("stays", ascending=False).head(40).round(3))
